# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata information
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references to entities use their `@id` as required by the Croissant schema.

In [ ]:
from pprint import pprint

# List record set @id's in this Croissant dataset
print("Available record sets (by @id):")
record_sets = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
for rsid in record_sets:
    print(f"  - {rsid}")

if not record_sets:
    print("No record sets found directly in the metadata. Attempting to infer from distributions...")
    # Fall back: try to infer record sets from distribution objects
    distributions = dataset.metadata.to_json().get('distribution', [])
    for d in distributions:
        if '@id' in d:
            print(f"  - {d['@id']}")
    record_sets = [d['@id'] for d in distributions if '@id' in d]

# For each record set, print available fields and columns by @id
for record_set_id in record_sets:
    print(f"\nExploring record set: {record_set_id}")
    try:
        recset_obj = dataset.metadata.find_by_id(record_set_id)
        fields = recset_obj.get('field', []) if recset_obj else []
        print("  Fields (@id):")
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                print(f"    - {f['@id']}")
            elif isinstance(f, str):
                print(f"    - {f}")
    except Exception as e:
        print(f"  (Could not access fields: {e})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All operations reference record set and field `@id`s from above.

In [ ]:
# List of record_set ids to extract records from
record_sets = record_sets  # as determined above
dataframes = {}

for rec_set_id in record_sets:
    print(f"\nExtracting records from record set: {rec_set_id}")
    try:
        records = list(dataset.records(record_set=rec_set_id))
        df = pd.DataFrame(records)
        dataframes[rec_set_id] = df
        print(f"Loaded {len(df)} records. Fields: {list(df.columns)}")
    except Exception as e:
        print(f"  Could not extract data for {rec_set_id}: {e}")

# Pick first available dataframe as main example (if present)
if dataframes:
    main_recset_id = list(dataframes.keys())[0]
    print(f"Using {main_recset_id} as main record set for downstream analysis.\n")
    print(dataframes[main_recset_id].head())
else:
    main_recset_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, or grouping. References use field `@id`s. (Modify to use a real numeric field from the dataset.)

In [ ]:
# You may need to adjust the field names according to what's present in the data
import numpy as np

# For demonstration: try to find a numeric field automatically
if main_recset_id and not dataframes[main_recset_id].empty:
    df = dataframes[main_recset_id]
    # Try to identify a numeric field to use
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        # try to coerce columns to numeric and use first that succeeds
        for col in df.columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notnull().sum() > 0:
                numeric_field = col
                df[col] = coerced
                break

    if numeric_field:
        print(f"Using numeric field '{numeric_field}' for EDA.")

        threshold = df[numeric_field].quantile(0.9)  # Outlier threshold as example
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (90th percentile):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean())
            / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical field
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped (mean of {numeric_field}) by '{group_field}':")
            print(grouped_df.head())
        else:
            print('No non-numeric field to group by.')
    else:
        print('No numeric fields found for EDA. Please check the data structure.')
else:
    print('Main DataFrame is empty or not available.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if main_recset_id and not dataframes[main_recset_id].empty and 'numeric_field' in locals() and numeric_field:
    df = dataframes[main_recset_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouped_df exists, plot a bar chart
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,5))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field])
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('No visualization: No numeric data available.')

## 6. Conclusion
This notebook demonstrated how to explore a Croissant dataset with multiple record sets using the `mlcroissant` library.

- We listed all available record sets and fields by their `@id`.
- We loaded tabular data from the main record set for analysis.
- We performed basic EDA with filtering and normalization via numeric field selection.
- We visualized data distributions to reveal potential outliers and trends.

**Next Steps:**
- Customize filtering, grouping, and visualization for your research question.
- Consult the dataset schema or Croissant documentation to further refine field mappings and semantic references.

For more, see: https://github.com/mlcommons/croissant/